# 🚀 02. Train Vision-Only Student Model (Phase 2 - CMKD)

> **Project:** Durian Ripeness Classification via Cross-Modal Knowledge Distillation (CMKD)  
> **Lead:** Leader / Member 1 (Vision Lead & Multi-Modal Framework)  
> **Reference:** `specification/project_specification.pdf` (Section 1.3.3) & `tmp_student.md`  
> **Notebook Role:**
1. Mount Google Drive & verify GPU availability.
2. `git clone` / `git pull` the latest source code from GitHub.
3. Link dataset (`data/final`), Teacher checkpoint (`checkpoints/teachermodal/best_teacher.pth`), and create target folders (`checkpoints/studentmodal`, `results_student`) on Google Drive.
4. Launch 2-phase distillation training (`train_student.py`) and export the benchmark comparison table.

In [1]:
# ==============================================================================
# 1. MOUNT GOOGLE DRIVE & CHECK GPU
# ==============================================================================
import os
import sys

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
if IN_COLAB:
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")
    print("✅ Google Drive mounted successfully!")
else:
    print("ℹ️ Running in Local environment.")

!nvidia-smi

Mounted at /content/drive
✅ Google Drive mounted successfully!
Sat Nov 21 02:58:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P0             43W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |         

In [2]:
# ==============================================================================
# 2. SYNC LATEST CODE FROM GITHUB
# ==============================================================================
REPO_URL = "https://github.com/Cocobaut/durian-ripeness-classification.git"
BRANCH = "cocobaut"
REPO_DIR = "/content/durian-ripeness-classification"

if IN_COLAB:
    %cd /content
    if not os.path.exists(REPO_DIR):
        if BRANCH:
            !git clone -b $BRANCH $REPO_URL $REPO_DIR
        else:
            !git clone $REPO_URL $REPO_DIR
        %cd $REPO_DIR
    else:
        %cd $REPO_DIR
        # Remove temporary repo symlinks (without touching Drive files) to prevent git pull conflicts
        !find . -maxdepth 5 -type l -delete
        !git fetch origin
        if BRANCH:
            !git checkout -f $BRANCH
            !git reset --hard origin/$BRANCH
        else:
            !git reset --hard origin/$(git rev-parse --abbrev-ref HEAD)
else:
    cur = os.path.abspath(".")
    while not (os.path.isdir(os.path.join(cur, "src")) and os.path.isdir(os.path.join(cur, "workspaces"))):
        parent = os.path.dirname(cur)
        if parent == cur:
            break
        cur = parent
    os.chdir(cur)

print(f"✅ Working in project root directory: {os.getcwd()}")

/content
Cloning into '/content/durian-ripeness-classification'...
remote: Enumerating objects: 1032, done.
remote: Counting objects: 100% (400/400), done.
remote: Compressing objects: 100% (279/279), done.
remote: Total 1032 (delta 137), reused 331 (delta 110), pack-reused 632 (from 1)
Receiving objects: 100% (1032/1032), 72.41 MiB | 48.50 MiB/s, done.
Resolving deltas: 100% (406/406), done.
/content/durian-ripeness-classification
✅ Working in project root directory: /content/durian-ripeness-classification


In [3]:
# ==============================================================================
# 3. FAST DEPENDENCY INSTALLATION WITH `uv`
# ==============================================================================
!pip install -q uv
!uv pip install --system -r requirements.txt
print("✅ Installed all dependencies successfully!")

Using Python 3.13.16 environment at: /usr
Resolved 184 packages in 831ms                                       
Prepared 15 packages in 490ms                                            
Uninstalled 3 packages in 20ms
Installed 15 packages in 109ms                              
 + async-lru==2.4.0
 + comm==0.2.3
 - ipykernel==6.17.1
 + ipykernel==7.4.0
 + jedi==0.20.1
 + json5==0.17.3
 + jupyter==1.1.1
 + jupyter-builder==1.3.0
 - jupyter-client==7.4.9
 + jupyter-client==8.10.0
 + jupyter-lsp==2.3.1
 + jupyterlab==4.6.6
 + jupyterlab-server==2.28.1
 + nest-asyncio2==1.7.4
 - notebook==6.5.7
 + notebook==7.6.5
 + pydeck==0.9.3
 + streamlit==1.67.0
✅ Installed all dependencies successfully!


In [4]:
# ==============================================================================
# 4. CONFIGURE CHECKPOINT & DATASET LINKS FROM GOOGLE DRIVE
# ==============================================================================
from pathlib import Path

if IN_COLAB:
    DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/durian-ripeness-cmkd")
    DRIVE_CHECKPOINTS = DRIVE_PROJECT_ROOT / "checkpoints"
    DRIVE_DATA = DRIVE_PROJECT_ROOT / "data" / "final"
    DRIVE_RESULTS_TEACHER = DRIVE_PROJECT_ROOT / "results_teacher"
    DRIVE_RESULTS_STUDENT = DRIVE_PROJECT_ROOT / "results_student"

    # Pre-create studentmodal & results_student folders on Google Drive for real-time persistence
    (DRIVE_CHECKPOINTS / "teachermodal").mkdir(parents=True, exist_ok=True)
    (DRIVE_CHECKPOINTS / "studentmodal").mkdir(parents=True, exist_ok=True)
    DRIVE_RESULTS_STUDENT.mkdir(parents=True, exist_ok=True)

    # Symlink repo directories to Google Drive so outputs persist across Colab disconnects
    !rm -rf checkpoints/unimodal checkpoints/teachermodal checkpoints/studentmodal
    !rm -rf workspaces/member1_vision_lead/04_train_teacher_and_student/results_student
    !mkdir -p checkpoints data/final
    !ln -sfn "{str(DRIVE_CHECKPOINTS / 'unimodal')}" checkpoints/unimodal
    !ln -sfn "{str(DRIVE_CHECKPOINTS / 'teachermodal')}" checkpoints/teachermodal
    !ln -sfn "{str(DRIVE_CHECKPOINTS / 'studentmodal')}" checkpoints/studentmodal
    !ln -sfn "{str(DRIVE_RESULTS_STUDENT)}" workspaces/member1_vision_lead/04_train_teacher_and_student/results_student
    if DRIVE_RESULTS_TEACHER.exists():
        !rm -rf workspaces/member1_vision_lead/04_train_teacher_and_student/results
        !ln -sfn "{str(DRIVE_RESULTS_TEACHER)}" workspaces/member1_vision_lead/04_train_teacher_and_student/results
    for split_name in ["train", "valid", "test"]:
        src_split = DRIVE_DATA / split_name
        if src_split.exists():
            !rm -rf data/final/{split_name}
            !ln -sfn "{str(src_split)}" data/final/{split_name}

Path("checkpoints/studentmodal").mkdir(parents=True, exist_ok=True)
print("✅ Verifying configured paths:")
print(" - Teacher ckpt   :", Path("checkpoints/teachermodal/best_teacher.pth").exists(), "->", Path("checkpoints/teachermodal").resolve())
print(" - Studentmodal   :", Path("checkpoints/studentmodal").resolve())
print(" - Results student:", Path("workspaces/member1_vision_lead/04_train_teacher_and_student/results_student").resolve())
print(" - Manifest CSV   :", Path("data/final/final_multimodal_manifest.csv").exists())

✅ Verifying configured paths:
 - Teacher ckpt   : True -> /content/drive/MyDrive/durian-ripeness-cmkd/checkpoints/teachermodal
 - Studentmodal   : /content/drive/MyDrive/durian-ripeness-cmkd/checkpoints/studentmodal
 - Results student: /content/drive/MyDrive/durian-ripeness-cmkd/results_student
 - Manifest CSV   : True


In [5]:
# ==============================================================================
# 5. TRAIN VISION-ONLY STUDENT (SEED 42) VIA CMKD v5 (INDEPENDENT VAL + FREE BACKBONE)
# ==============================================================================
# CMKD v5 (Direction 2): allow ConvNeXtV2-Base backbone to genuinely adapt beyond the unimodal
# local minimum while using held-out Validation for honest Early Stopping & checkpoint selection:
#  1. --train-splits train + --select-metric val_loss + --patience 10: held-out Val (24 fruits)
#     selects the lowest continuous val_loss (so Phase 2 checkpoints are not blocked by frozen Phase 1 F1).
#  2. --lr-backbone 2e-5, --lr 2e-4 (no L2-SP, no EMA lag): unblocks backbone adaptation so
#     ConvNeXtV2 can learn new visual cues from Teacher instead of only rotating the linear head.
#  3. Retains CMKD v4 fixes: --dedup-fruits (unbiased 1 sample/fruit & true fruit class weights),
#     CE + label smoothing, relational fused feature alignment, teacher_confidence KD, Feature Mixup.
SEED = 42
!python workspaces/member1_vision_lead/04_train_teacher_and_student/train_student.py \
    --manifest data/final/final_multimodal_manifest.csv \
    --data-dir data/final \
    --teacher-ckpt checkpoints/teachermodal/best_teacher.pth \
    --student-backbone convnextv2_base \
    --save-path checkpoints/studentmodal/best_student.pth \
    --results-dir workspaces/member1_vision_lead/04_train_teacher_and_student/results_student \
    --train-splits train \
    --dedup-fruits \
    --loss-type ce --label-smoothing 0.08 \
    --alpha 1.0 --beta 0.5 --gamma 0.25 --temperature 3.0 \
    --feat-target fused --feat-loss-type relational --kd-weighting teacher_confidence \
    --mixup-alpha 0.2 --mixup-prob 0.5 \
    --l2-sp-weight 0.0 \
    --ema-decay 0.0 \
    --batch-size 4 --accum-steps 2 \
    --epochs 25 --warmup-epochs 2 --patience 10 \
    --lr 2e-4 --lr-backbone 2e-5 --min-lr 1e-6 \
    --select-metric val_loss \
    --eval-test \
    --fresh \
    --seed {SEED} \
    --device cuda

[2026-11-21 02:59:22] ==============================================================================
[2026-11-21 02:59:22] TRAINING VISION-ONLY STUDENT VIA CMKD (STAGE 2)
[2026-11-21 02:59:22] Device: cuda | AMP: True (torch.bfloat16) | batch 4 x accum 2
[2026-11-21 02:59:22] Student backbone : convnextv2_base.fcmae_ft_in1k
[2026-11-21 02:59:22] Distill weights  : alpha=1.0 (CE) | beta=0.5 (KD, T=3.0) | gamma=0.25 (Feat: relational)
[2026-11-21 02:59:22] Best student ckpt: /content/durian-ripeness-classification/checkpoints/studentmodal/best_student.pth
[2026-11-21 02:59:22] ==============================================================================
[2026-11-21 02:59:22] Loading Frozen Teacher from: /content/durian-ripeness-classification/checkpoints/teachermodal/best_teacher.pth
[2026-11-21 02:59:30] L_feat target: Teacher 'fused' feature (256-d) | KD weighting: teacher_confidence | EMA: 0.985 | TTA: True
/content/durian-ripeness-classification/src/datasets/sensor_dataset.py:132: U

In [6]:
# ==============================================================================
# 6. INDEPENDENT TEST EVALUATION & BENCHMARK COMPARISON TABLE
# ==============================================================================
RES = Path("workspaces/member1_vision_lead/04_train_teacher_and_student/results_student")

!python workspaces/member1_vision_lead/04_train_teacher_and_student/eval_student.py \
    --checkpoint checkpoints/studentmodal/best_student.pth \
    --manifest data/final/final_multimodal_manifest.csv \
    --data-dir data/final \
    --split test \
    --seed {SEED} \
    --output-dir {str(RES)}

comp_md = RES / "benchmark_comparison.md"
if comp_md.is_file():
    print("\n" + comp_md.read_text(encoding="utf-8"))
print("✅ All Student evaluation results saved at:", RES.resolve())

[2026-11-21 03:23:35] [INFO] [eval_student]: Student evaluation on 'test' | TTA: True
[2026-11-21 03:23:35] [INFO] [eval_teacher]: Initialized TeacherEvaluator on device: cuda (input_mode=vision)
[2026-11-21 03:23:35] [INFO] [eval_teacher]: Evaluation results will be saved to: /content/durian-ripeness-classification/workspaces/member1_vision_lead/04_train_teacher_and_student/results_student
[2026-11-21 03:23:35] [INFO] [eval_teacher]: === STARTING FULL TEACHER EVALUATION PIPELINE ===
[2026-11-21 03:23:35] [INFO] [eval_teacher]: model=vision_student | split=test | input_mode=vision | strategy=mean_prob
[2026-11-21 03:23:35] [INFO] [eval_teacher]: Starting sample-level evaluation (mask_modality=None)...
[2026-11-21 03:23:48] [INFO] [eval_teacher]: Evaluated 24 samples, accuracy = 0.5000
[2026-11-21 03:23:48] [INFO] [eval_teacher]: Aggregating fruit-level predictions (fruit_id) using strategy: mean_prob...
[2026-11-21 03:23:48] [INFO] [eval_teacher]: Aggregated 24 fruits, fruit-level accu